## Project: Mapping the Potential Destructive Power of Wildfires Using Machine Learning

### Module 3: *Feature Engineering*

---
### Contents  
- 1. Construct Target Variable (Low, Moderate, High) based on estimated damages cost
- 2. Independent Variables Analysis
- 3. Feature Engineering
- 4. Feature Selection and Correlation Analysis
---
### Notes
---
### Inputs
- `merged_weather_fire` cleaned weather data joined with cleaned fire damage dataset
---
### Outputs 
- `X`,`y` - for future modeling
- `details` - for future plotting
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

# Function to print a grid of kde plots in consistent format, adjusts columns and rows accordingly
from src.plot_utils import grid_kde

# Function to print a custom format correlation heatmap
from src.plot_utils import correlation_map

# Function to create all second degree interactions possible in a dataset, then name these terms
# in the format of feature1_x_feature2. Returns dataframe of interactions.
from src.data_utils import create_2nd_degree_interactions

# A space saving function to rank interactions
from src.data_utils import rank_interactions_by_correlation

# Function to calculate dryness index and return a dataframe
from src.data_utils import calculate_dryness_index

---
### Third Party Dependencies

In [2]:
# Core data tools
import pandas as pd
import numpy as np
from datetime import datetime

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats.mstats import winsorize

# Modeling prep
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.preprocessing import PolynomialFeatures

# Set consistent plotting style
sns.set(style="whitegrid")
plt.rcParams["figure.dpi"] = 100

---

### Load Datasets

In [3]:
# Load cleaned main dataset
final = pd.read_csv("../data/processed/selected.csv")

In [4]:
final

,Unnamed: 0,Lat,Lon,station_dist,Avg Air Temp (F),Avg Rel Hum (%),Avg Wind Speed (mph),Avg Soil Temp (F),Precip (in),Avg Vap Pres (mBars),Sol Rad (Ly/day),ETo (in),Total Population,density,Mean Income,Target
0,0,33.973814,-117.430053,0.093492,65.614286,22.857143,4.914286,61.142857,0.00,4.642857,429.428571,1.13,2492442,345.861225,93563,1
1,1,37.320320,-121.754095,0.367749,67.485714,63.571429,4.028571,72.885714,0.00,14.542857,709.714286,1.72,1877592,1455.384854,174331,0
2,2,37.992816,-120.661811,0.332019,72.500000,60.714286,4.614286,68.342857,0.00,16.857143,701.857143,1.83,46565,45.651513,84164,1
3,4,33.656392,-117.443056,0.146400,82.185714,52.857143,1.314286,75.828571,0.00,19.385714,535.571429,1.39,3135755,3966.448259,127056,2
4,5,34.505960,-120.083681,0.077311,61.742857,67.857143,2.928571,75.157143,0.13,12.671429,408.571429,0.92,441257,161.331803,111731,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
193,221,39.376113,-121.326385,0.124016,78.271429,46.285714,3.100000,76.442857,0.00,15.285714,477.285714,1.39,85722,135.670423,73030,2
194,222,39.319278,-121.276206,0.077515,83.471429,31.285714,4.057143,75.885714,0.00,12.171429,715.571429,2.11,85722,135.670423,73030,1
195,224,37.969815,-120.400392,0.515830,73.600000,61.000000,4.800000,72.042857,0.00,17.271429,566.285714,1.50,54204,24.406542,83401,0
196,225,34.077969,-118.818547,0.193524,66.228571,52.000000,3.728571,66.857143,0.00,10.542857,320.571429,0.88,9663345,2381.377714,103220,2


---

## 3. Additional Features

### Days Without Rain

flag to record whether **signifigant ( >1 inch)** precipitation has occured on a day

In [5]:
stations = final['Stn Id'].unique()

final['Days Without Rain'] = 0
day_count = 0;

for station in stations:
    station_indexes = final[final['Stn Id'] == station].index
    
    for index, row in final.loc[station_indexes].iterrows():
        if row['Precip (in)'] > 0:
            day_count = 0
        else:
            day_count = day_count + 1
    
        final.at[index, 'Days Without Rain'] = day_count


KeyError: 'Stn Id'

In [ ]:
final['Days Without Rain']

### Historical fires per month

Create a new column that records the historical average number of fires for each month, based on all previous years up to that point. Each row should reflect the average number of fires that occurred in the same calendar month across prior years, providing a historical baseline for comparison.

**ASSUMPTION** -Since the weather coverage in this dataset is limited, no previous year readings exists.
For now, I will just use the current count as a rough estimation for the average. 

In [ ]:
final['Date'] = pd.to_datetime(final['Date'])

# Convert Datetime column to string
final['Month_Year'] = final['Date'].dt.strftime('%Y-%m')
month_year = final['Month_Year'].unique()

In [ ]:
# Dictionary to store month-to-count mapping
monthly_fire_counts = {}

# Step 1: Calculate fire counts for each month
for month in month_year:
    # Get index and subset
    month_index = final[final['Month_Year'] == month].index
    subset = final.loc[month_index]
    
    # Eliminate 'Low' days
    total = subset[subset['Target'] != 'Low']
    
    # Store count
    monthly_fire_counts[month] = len(total)

# Step 2: Calculate 2-year rolling average
# Convert month_year to sorted list to ensure order
month_year_sorted = sorted(month_year)

# Dictionary for rolling averages
rolling_avg = {}

for i, month in enumerate(month_year_sorted):
    if i == 0:
        # First year: average of all *future* years (exclude current)
        future_values = list(monthly_fire_counts.values())[1:]
        rolling_avg[month] = np.mean(future_values) if future_values else 0

    elif i == 1:
        # Second year: average of first year and current
        prev = month_year_sorted[0]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev], monthly_fire_counts[month]])
    
    else:
        # Rolling average of previous 2 years
        prev1 = month_year_sorted[i - 1]
        prev2 = month_year_sorted[i - 2]
        rolling_avg[month] = np.mean([monthly_fire_counts[prev1], monthly_fire_counts[prev2]])

# Step 3: Assign back to each row in the dataframe
for month in month_year_sorted:
    month_index = final[final['Month_Year'] == month].index
    final.loc[month_index, '2-Year Avg Fires'] = rolling_avg[month]


In [ ]:
drop = ['Month_Year']

### 3.1 Dryness Indicator

PET_proxy = a * T_mean + b * Wind + c * SolarRadiation - d * Humidity

In [ ]:
final['Dryness'] = calculate_dryness_index(final).astype(float)

#final['Dryness'] = pd.qcut(final['Dryness'], q=3, labels=[0, 1, 2])

In [ ]:
grid_kde(final[['Dryness']])

### 3.2 Add Lagged Weather Variables

We calculate 7-day rolling averages for select weather variables to capture recent trends that may influence fire severity.

In [ ]:
# Sort data by County and Date to prepare for rolling average
final = final.sort_values(by=['County', 'Date'])

# Define columns for 7-day rolling average
avg_columns = [
    'Avg Air Temp (F)', 
    'Precip (in)',
    'Avg Rel Hum (%)', 
    'Avg Wind Speed (mph)'
]

# Apply rolling mean by County
for col in avg_columns:
    final[f'{col} 7 Day Avg'] = final.groupby('County')[col].rolling(window=7, min_periods=1).mean().reset_index(level=0, drop=True)

## 4.4 Feature Interaction Analysis

In [ ]:
drop = ['Month_Year','PET_proxy']
final = final.drop(columns=drop)

In [ ]:
final = final.sort_values(by='Date')

In [ ]:
final = final.reset_index(drop=True)

In [ ]:
final.to_csv('../data/processed/unscaled_X.csv')

In [ ]:
final.info()

In [ ]:
y = final['Target']
X = final.drop(columns = ['Duration','Adjusted Value','Target',
                                  'Stn Id','Latitude','Longitude',
                                  'Date','County','Adjusted Value'])
details = final[['Duration','Adjusted Value','Target',
                                  'Stn Id','Latitude','Longitude',
                                  'Date','County','Adjusted Value']]

Scale all datasets and save back to dataframe format

In [ ]:
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)  # Scales and returns a NumPy array
X_scaled = pd.DataFrame(X_scaled, columns=X.columns, index=X.index)

For all independent variables create a new dataframe containing every possible interaction of 2nd degree

In [ ]:
inter_X = create_2nd_degree_interactions(X_scaled)

Combine interaction terms with single variables for correlation analysis

In [ ]:
inter_X_combined = pd.concat([X_scaled, inter_X], axis=1)

### Interaction Correlation analysis

In [ ]:
correlation_results = rank_interactions_by_correlation(inter_X_combined, y)

In [ ]:
keep = correlation_results.head(40)['Feature'].tolist()

In [ ]:
correlation_results.head(40)

Keep top 20 results

In [ ]:
top_20 = inter_X_combined[keep]

# Compute correlation matrix
corr_matrix = top_20.corr()

# Select upper triangle of correlation matrix
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Find columns with correlation > 0.85
to_drop = [column for column in upper.columns if any(upper[column] > 0.85)]

# Drop those columns
df_reduced = top_20.drop(columns=to_drop)

print("Dropped columns:", to_drop)
print("Reduced DataFrame:")

df_reduced

In [ ]:
X = df_reduced

## Export Feature Data

In [ ]:
X.to_csv('../data/processed/X.csv', index=False)
y.to_csv('../data/processed/y.csv', index=False)
details.to_csv('../data/processed/details.csv')
print("All datasets saved successfully to ../data/processed/")

In [ ]:

X